# Recreate NZ Comparison Charts

This notebook recreates the FRP/MOp comparison charts from the manuscript method text.

Workflow:
1. Start from original normalized Z-score vectors.
2. Build enhanced vectors with natural-log biasing and L2 renormalization.
3. Transform displayed values with `log10(1 + 8 * NZ)`.
4. Save `comparison_FRP_chart.svg` and `comparison_MOp_chart.svg`.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import matplotlib.pyplot as plt
import numpy as np

OUTPUT_DIR = Path("/home/sy/motif/motif_code")

In [ ]:
MOP_NZ = np.array(
    [
        -0.00722737,
        -0.01273873,
        -0.01561858,
        0.01562969,
        0.00866638,
        0.00402705,
        0.06961442,
        0.06682548,
        0.02137642,
        -0.00520205,
        0.02198735,
        0.23077471,
        0.96730022,
    ],
    dtype=np.float64,
)

FRP_NZ = np.array(
    [
        -0.01928962,
        0.20980253,
        -0.03922592,
        0.18195760,
        0.00601686,
        0.03389555,
        0.09883095,
        0.48222753,
        0.01604604,
        -0.01396816,
        0.02385361,
        0.27786690,
        0.77410329,
    ],
    dtype=np.float64,
)

COLORS = {
    "MOp_E": "#601986",
    "MOp": "#F3CC4F",
    "FRP_E": "#E60012",
    "FRP": "#F18D00",
}

In [ ]:
def make_enhanced_nz(nz, pos_scale=5.0, neg_scale=3.0):
    enhanced = nz.astype(np.float64).copy()
    pos = nz > 0
    neg = nz < 0

    enhanced[pos] = np.log1p(pos_scale * nz[pos])
    enhanced[neg] = -np.log1p(neg_scale * np.abs(nz[neg]))

    norm = np.linalg.norm(enhanced)
    if norm == 0:
        raise ValueError("Cannot normalize an all-zero vector.")
    return enhanced / norm


def visualization_transform(nz):
    argument = 1.0 + 8.0 * nz
    if np.any(argument <= 0):
        bad = nz[argument <= 0]
        raise ValueError(f"log10(1 + 8 * NZ) is undefined for values: {bad}")
    return np.log10(argument)


def plot_pair(first, second, first_label, second_label, output_path):
    index = np.arange(1, 14)
    bar_width = 0.3

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(index, first, bar_width, label=first_label, color=COLORS[first_label])
    ax.bar(index + bar_width, second, bar_width, label=second_label, color=COLORS[second_label])

    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.set_xticks(index + bar_width)
    ax.set_xticklabels(range(1, 14))
    ax.set_ylim(-0.2, 1)
    ax.legend()
    ax.set_title("Comparison of MOp, FRP and their Average")
    ax.set_xlabel("Index")
    ax.set_ylabel("Value")

    fig.tight_layout()
    fig.savefig(output_path, format="svg", dpi=300, bbox_inches="tight")
    return fig, ax

In [ ]:
MOP_ENHANCED = make_enhanced_nz(MOP_NZ)
FRP_ENHANCED = make_enhanced_nz(FRP_NZ)

MOP_TRANSFORMED = visualization_transform(MOP_NZ)
MOP_E_TRANSFORMED = visualization_transform(MOP_ENHANCED)
FRP_TRANSFORMED = visualization_transform(FRP_NZ)
FRP_E_TRANSFORMED = visualization_transform(FRP_ENHANCED)

print("FRP_E transformed:", np.array2string(FRP_E_TRANSFORMED, precision=8))
print("FRP transformed:", np.array2string(FRP_TRANSFORMED, precision=8))
print("MOp_E transformed:", np.array2string(MOP_E_TRANSFORMED, precision=8))
print("MOp transformed:", np.array2string(MOP_TRANSFORMED, precision=8))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plot_pair(
    FRP_E_TRANSFORMED,
    FRP_TRANSFORMED,
    "FRP_E",
    "FRP",
    OUTPUT_DIR / "comparison_FRP_chart.svg",
)

plot_pair(
    MOP_E_TRANSFORMED,
    MOP_TRANSFORMED,
    "MOp_E",
    "MOp",
    OUTPUT_DIR / "comparison_MOp_chart.svg",
)

print(f"Saved charts to {OUTPUT_DIR}")